# 🎬 AI YouTube Automation — Video Engine

**AI-powered YouTube video & Shorts generation engine**

This notebook runs the full pipeline: from a single motivational quote to a fully-rendered YouTube video — **no API tokens required!**

| Step | Module | What it does |
|------|--------|--------------|
| 1 | Story Generation | Creates a unique story using Ollama LLM (Gemma3) |
| 2 | SEO Metadata | Generates title, description, and hashtags |
| 3 | Image Prompt | Summarises the story into a visual prompt |
| 4 | Image Generation | Creates images locally using **Stable Diffusion XL** on GPU |
| 5 | Audio Generation | Converts story to speech (Kokoro TTS) |
| 6 | Transcription | Generates subtitles from audio (Whisper) |
| 7 | Subtitle Processing | Converts SRT → JSON for MoviePy |
| 8 | Video Assembly | Renders landscape video + YouTube Shorts |
| 9 | YouTube Upload | OAuth2 upload (skipped in Colab by default) |

> ⚡ **GPU Required**: Select **T4 GPU** from `Runtime > Change runtime type`

---

## 📦 Step 1 — Install System Dependencies

Install `ffmpeg`, `espeak-ng`, `ImageMagick` policy fix, and `Ollama`.

**These are required** — without them, audio generation, video rendering, and subtitles will fail.

In [ ]:
%%bash
# ═══════════════════════════════════════════════
# System Dependencies (DO NOT SKIP)
# ═══════════════════════════════════════════════

# 1. FFmpeg + zstd — required for video rendering and Ollama install
apt-get update -qq && apt-get install -y -qq ffmpeg zstd imagemagick > /dev/null 2>&1
echo "✅ ffmpeg installed"
echo "✅ zstd installed"

# 2. espeak-ng — required for Kokoro TTS
apt-get install -y -qq espeak-ng > /dev/null 2>&1
echo "✅ espeak-ng installed"

# 3. Fix ImageMagick policy — required for MoviePy TextClip subtitles
POLICY_FILE="/etc/ImageMagick-6/policy.xml"
if [ -f "$POLICY_FILE" ]; then
    # Remove the restrictive policy that blocks text rendering
    sed -i 's/<policy domain="path" rights="none" pattern="@\*"/<policy domain="path" rights="read|write" pattern="@*"/' "$POLICY_FILE"
    echo "✅ ImageMagick policy fixed"
else
    echo "⚠️  ImageMagick policy file not found (may not need fixing)"
fi

# 4. Install Ollama
if curl -fsSL https://ollama.ai/install.sh | sh 2>/dev/null; then
    echo "✅ Ollama installed"
else
    echo "❌ Ollama installation failed — re-run this cell"
    exit 1
fi

## 📥 Step 2 — Clone Repository & Install Python Dependencies

In [ ]:
import os

REPO_URL = "https://github.com/tamilarasu18/ai-youtube-automation.git"
REPO_DIR = "/content/ai-youtube-automation"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository exists — pulling latest...")
    !cd {REPO_DIR} && git pull --ff-only

%cd {REPO_DIR}

# Install Python dependencies
!pip install -q -r requirements.txt
!pip install -q -e .

print("\n✅ Python dependencies installed!")


## 🚀 Step 3 — Start Ollama & Pull the LLM Model

> ⏳ **First run**: Model download takes ~3–5 minutes.

In [ ]:
import subprocess
import time

# Start Ollama server in background
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)
print("✅ Ollama server started (PID: {})".format(process.pid))

# Pull the Gemma 3 model
print("\n📥 Downloading gemma3:4b model...")
!ollama pull gemma3:4b
print("\n✅ Model ready!")

## ⚙️ Step 4 — Configure Environment

**No API tokens needed!** Everything runs locally on the Colab GPU.

Adjust settings below if needed:

In [ ]:
import os
import torch

# ════════════════════════════════════════════════════════════
# ⚙️  Pipeline Configuration (all defaults work out of box)
# ════════════════════════════════════════════════════════════

# --- LLM ---
os.environ["OLLAMA_URL"] = "http://localhost:11434/api/generate"
os.environ["OLLAMA_MODEL"] = "gemma3:4b"

# --- Image Generation (Stable Diffusion XL — local GPU) ---
os.environ["SD_MODEL_ID"] = "stabilityai/stable-diffusion-xl-base-1.0"
os.environ["SD_NUM_STEPS"] = "25"       # 20-50 (lower = faster)
os.environ["SD_GUIDANCE_SCALE"] = "7.5"

# --- Audio ---
os.environ["KOKORO_VOICE"] = "af_heart"

# --- Transcription ---
os.environ["WHISPER_MODEL"] = "small.en"  # Use 'small' for Colab (saves VRAM)
os.environ["WHISPER_CACHE_DIR"] = "/content/whisper_cache"

# --- Video ---
os.environ["VIDEO_FPS"] = "30"
os.environ["VIDEO_BITRATE"] = "20000k"
os.environ["MAX_SHORTS_DURATION"] = "60"

# --- YouTube (skip upload in Colab) ---
os.environ["SKIP_UPLOAD"] = "false"

# --- Paths ---
os.environ["OUTPUT_DIR"] = "output"
os.environ["ASSETS_DIR"] = "assets"
os.environ["FONT_PATH"] = "assets/fonts/Anton-Regular.ttf"
os.environ["IMAGEMAGICK_BINARY"] = "/usr/bin/convert"
os.environ["BACKGROUND_MUSIC"] = "assets/audio/background_music.wav"
os.environ["LOG_FILE"] = "logs/colab.log"
os.environ["LOG_LEVEL"] = "INFO"

# ═══════════════════════════════════════════════════════════
# Status check
# ═══════════════════════════════════════════════════════════
gpu_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if gpu_available else "None"
gpu_mem = f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if gpu_available else "N/A"

from pathlib import Path
font_ok = Path(os.environ["FONT_PATH"]).exists()
os.environ["IMAGEMAGICK_BINARY"] = "/usr/bin/convert"
music_ok = Path(os.environ["BACKGROUND_MUSIC"]).exists()

print("✅ Configuration set!")
print(f"\n🖥️  GPU: {gpu_name} ({gpu_mem})")
print(f"🤖 LLM: {os.environ['OLLAMA_MODEL']}")
print(f"🎨 Image: Stable Diffusion XL ({os.environ['SD_NUM_STEPS']} steps, 1024×1024 native)")
print(f"🗣️  Voice: {os.environ['KOKORO_VOICE']}")
print(f"📝 Whisper: {os.environ['WHISPER_MODEL']}")
print(f"🔤 Font: {'✅' if font_ok else '❌ Missing!'} {os.environ['FONT_PATH']}")
print(f"🎵 Music: {'✅' if music_ok else '⚠️  Missing'} {os.environ['BACKGROUND_MUSIC']}")
print(f"📤 Upload: {'Skipped' if os.environ['SKIP_UPLOAD'] == 'true' else 'Enabled'}")

if not gpu_available:
    print("\n⚠️  WARNING: No GPU detected!")
    print("   Go to Runtime > Change runtime type > Select T4 GPU")

## 🔑 Step 4b — Load YouTube Credentials from Google Drive

Mounts Google Drive and loads your YouTube credentials automatically.

> No manual OAuth needed — uses your existing refresh token from `config.json`


In [ ]:
import os, json, shutil
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive", force_remount=False)

DRIVE_DIR = "/content/drive/MyDrive/ai-youtube-automation"
os.makedirs("config", exist_ok=True)

# ── Build credentials from existing config.json ──
drive_config = os.path.join(DRIVE_DIR, "config.json")
drive_client_secret = os.path.join(DRIVE_DIR, "client_secret.json")

if os.path.exists(drive_config):
    cfg = json.loads(open(drive_config).read())

    # Save client_secret.json (for any future re-auth if needed)
    if os.path.exists(drive_client_secret):
        shutil.copy2(drive_client_secret, "config/client.json")
        print("✅ client.json copied from Drive")

    # Build token.json from existing refresh token
    token_data = {
        "token": None,
        "refresh_token": cfg.get("youtube_refresh_token"),
        "token_uri": "https://oauth2.googleapis.com/token",
        "client_id": cfg.get("youtube_client_id"),
        "client_secret": cfg.get("youtube_client_secret"),
        "scopes": ["https://www.googleapis.com/auth/youtube.upload"],
    }

    with open("config/token.json", "w") as f:
        json.dump(token_data, f, indent=2)
    print("✅ token.json built from existing refresh token")
    print("   YouTube upload is enabled — no OAuth prompt needed!")
else:
    print(f"❌ config.json not found in {DRIVE_DIR}")
    print("   Upload will be skipped.")


## 🎯 Step 5 — Run the Full Pipeline

Enter your motivational quote below and run the cell.

The video will be **scheduled to publish tomorrow at 10:00 AM IST** on YouTube.

> ⏳ **Estimated time**: 8–15 minutes (first run downloads SDXL model ~6.5GB)

> 🔐 **First-time YouTube upload**: You'll be prompted to authorize via a URL. Click the link, sign in, and paste the code back.

In [ ]:
import sys, os

# Ensure video_engine is importable (src layout)
REPO_SRC = "/content/ai-youtube-automation/src"
if os.path.isdir(REPO_SRC) and REPO_SRC not in sys.path:
    sys.path.insert(0, REPO_SRC)
if os.path.isdir("/content/ai-youtube-automation"):
    os.chdir("/content/ai-youtube-automation")

# ════════════════════════════════════════════════════════════
# ✏️ Enter your motivational quote / prompt here:
# ════════════════════════════════════════════════════════════
PROMPT = "The greatest glory in living lies not in never falling, but in rising every time we fall."

# ════════════════════════════════════════════════════════════
# 🚀 Run the pipeline
# ════════════════════════════════════════════════════════════
from video_engine.core.logger import setup_logging
from video_engine.core.pipeline import Pipeline

setup_logging()
pipeline = Pipeline()
# ════════════════════════════════════════════════════════════
# ⏰ Schedule upload for tomorrow 10:00 AM IST
#    Set to None to upload immediately (or skip upload entirely)
# ════════════════════════════════════════════════════════════
import pytz
from datetime import datetime, timedelta

ist = pytz.timezone("Asia/Kolkata")
tomorrow_10am = (datetime.now(ist) + timedelta(days=1)).replace(
    hour=10, minute=0, second=0, microsecond=0
)
SCHEDULED_TIME = tomorrow_10am.isoformat()
print(f"📅 Scheduled publish: {SCHEDULED_TIME}")

result = pipeline.run(prompt=PROMPT, scheduled_time=SCHEDULED_TIME)

# Print results
print("\n" + "═" * 60)
if result["success"]:
    print("🎉 PIPELINE COMPLETED SUCCESSFULLY!")
else:
    print(f"❌ PIPELINE FAILED: {result['error']}")
print("═" * 60)
print(f"Total time: {result['total_duration_s']:.1f}s")
print("\nStep breakdown:")
for step in result.get("steps", []):
    icon = "✅" if step["success"] else "❌"
    print(f"  {icon} {step['step']} — {step['duration_s']:.1f}s")

## ▶️ Step 6 — Preview Generated Videos

In [ ]:
from IPython.display import Video, display
from pathlib import Path

output_dir = Path("output")
videos = list(output_dir.rglob("*.mp4"))

if not videos:
    print("⚠️  No videos found — run the pipeline first.")
else:
    print(f"🎬 Found {len(videos)} video(s):\n")
    for v in videos:
        print(f"📹 {v}")
        try:
            display(Video(str(v), embed=True, width=480))
        except Exception:
            print("   (Preview not available — download to view)")
        print()

## 📥 Step 7 — Download Generated Files

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

output_dir = Path("output")
zip_name = "generated_videos.zip"

output_files = [f for f in output_dir.rglob("*") if f.is_file()]

if not output_files:
    print("⚠️  No output files found.")
else:
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in output_files:
            zf.write(f, f.relative_to(output_dir))
            print(f"  📎 {f.relative_to(output_dir)}")

    print(f"\n📦 {zip_name} — downloading...")
    files.download(zip_name)

---

## 🔄 Run Individual Steps (Optional)

### 📝 Generate Story Only

In [ ]:
import sys, os

# Ensure video_engine is importable (src layout)
REPO_SRC = "/content/ai-youtube-automation/src"
if os.path.isdir(REPO_SRC) and REPO_SRC not in sys.path:
    sys.path.insert(0, REPO_SRC)
if os.path.isdir("/content/ai-youtube-automation"):
    os.chdir("/content/ai-youtube-automation")

REPO_DIR = "/content/ai-youtube-automation"
if os.path.isdir(REPO_DIR) and REPO_DIR not in sys.path:
    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

from video_engine.core.config import get_settings
from video_engine.generators.story import generate_story

settings = get_settings()
settings.ensure_directories()

story = generate_story("The only way to do great work is to love what you do.", settings)
print("\n" + "═" * 50)
print(story)
print("═" * 50)

### 🎨 Generate Image Only

In [ ]:
import sys, os

# Ensure video_engine is importable (src layout)
REPO_SRC = "/content/ai-youtube-automation/src"
if os.path.isdir(REPO_SRC) and REPO_SRC not in sys.path:
    sys.path.insert(0, REPO_SRC)
if os.path.isdir("/content/ai-youtube-automation"):
    os.chdir("/content/ai-youtube-automation")

REPO_DIR = "/content/ai-youtube-automation"
if os.path.isdir(REPO_DIR) and REPO_DIR not in sys.path:
    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

from video_engine.core.config import get_settings
from video_engine.generators.image import generate_images, unload_model
from pathlib import Path
from IPython.display import display
from PIL import Image

settings = get_settings()
work_dir = settings.video_output_dir
work_dir.mkdir(parents=True, exist_ok=True)

# Write a prompt if not exists
prompt_path = work_dir / "prompt.txt"
if not prompt_path.exists():
    prompt_path.write_text(
        "A child standing on a rooftop gazing at a starlit sky, feeling hopeful",
        encoding="utf-8"
    )

generate_images(work_dir, settings)
unload_model()  # Free GPU memory

bg_dir = Path(settings.OUTPUT_DIR) / "background_file"
for img_file in sorted(bg_dir.glob("*.jpg")):
    print(f"\n🖼️  {img_file.name}")
    display(Image.open(img_file))

### 🧹 Free GPU Memory

In [ ]:
import sys, os

# Ensure video_engine is importable (src layout)
REPO_SRC = "/content/ai-youtube-automation/src"
if os.path.isdir(REPO_SRC) and REPO_SRC not in sys.path:
    sys.path.insert(0, REPO_SRC)
if os.path.isdir("/content/ai-youtube-automation"):
    os.chdir("/content/ai-youtube-automation")

REPO_DIR = "/content/ai-youtube-automation"
if os.path.isdir(REPO_DIR) and REPO_DIR not in sys.path:
    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

from video_engine.generators.image import unload_model
import torch, gc

unload_model()
gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"✅ GPU: {free/1e9:.1f} / {total/1e9:.1f} GB free")
else:
    print("✅ Memory cleaned")

---

## 📋 Troubleshooting

| Issue | Solution |
|-------|----------|
| `No GPU detected` | `Runtime > Change runtime type > T4 GPU` |
| `CUDA out of memory` | Run 🧹 Free GPU Memory cell, or set `WHISPER_MODEL=tiny` + `SD_NUM_STEPS=20` |
| `espeak not found` | Re-run Step 1 |
| `Ollama request failed` | Re-run Step 3 |
| `Ollama installation failed` | Re-run Step 1 — ensures `zstd` is installed first |
| `TextClip error / policy.xml` | Re-run Step 1 (ImageMagick fix) |
| `Font not found` | Font is included in repo at `assets/fonts/Anton-Regular.ttf` |
| Video won't preview | Download the zip and play locally |

---

**Repository**: [github.com/tamilarasu18/ai-youtube-automation](https://github.com/tamilarasu18/ai-youtube-automation)  
**License**: MIT